In [1]:
import pandas as pd
import csv
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

In [2]:
# Read in CA data and re-format date to the correct format
CA = pd.read_csv('../Data/california.csv', delimiter=",")
CA.rename(columns={'date': 'time'}, inplace=True)
CA['time'] = pd.to_datetime(CA['time'])
CA['date'] = CA['time'].dt.date
display(CA.head(3))

# Agg VADAR score based on business id and dates
agg_CA=CA.groupby(['business_id', 'date'])['doc_sentiment'].agg(avg_VADAR='mean').reset_index()
agg_CA = agg_CA.merge(CA, on=['business_id', 'date'])[['business_id','business_name','city','state','date','avg_VADAR']]
display(agg_CA.head(3))

print('Total Biz IDs:',len(agg_CA['business_id'].unique()))

,business_id,business_name,city,state,latitude,longitude,stars,total_review_count,review_id,rating,time,review,doc_sentiment,aspect_sentiments,date
0,U3grYFIeu6RgAAQgdriHww,Brophy Bros - Santa Barbara,Santa Barbara,CA,34.403759,-119.693992,4.0,2940,7Dcrt0Oz0hikA8obGhadrQ,5.0,2005-03-01 16:57:17,Words cannot describe how much I miss this pla...,0.9807,"{'PRODUCT': {}, 'PERSON': {}, 'ORG': {}}",2005-03-01
1,Aes-0Q_guDeYewMapFs_vg,Longboard's Grill,Santa Barbara,CA,34.410292,-119.686542,3.0,516,WC9q5vhQlQkLK05kEs-vYQ,2.0,2005-03-01 16:59:37,Food is decent but is a complete tourist trap....,0.8573,"{'PRODUCT': {}, 'PERSON': {}, 'ORG': {}}",2005-03-01
2,CziOtnFSklimJnBgksDDwA,Pascucci,Santa Barbara,CA,34.419107,-119.699243,3.5,1069,Q0GJ06L78nkVyNfHrd9iUg,3.0,2005-03-01 17:25:13,"This place gets a 3.5...great location, great ...",0.7220,"{'PRODUCT': {}, 'PERSON': {}, 'ORG': {}}",2005-03-01


,business_id,business_name,city,state,date,avg_VADAR
0,--O3ip9NpXTKD4oBS1pY2A,Alameda Park,Santa Barbara,CA,2007-08-14,0.8729
1,--O3ip9NpXTKD4oBS1pY2A,Alameda Park,Santa Barbara,CA,2007-10-23,0.9924
2,--O3ip9NpXTKD4oBS1pY2A,Alameda Park,Santa Barbara,CA,2007-11-20,0.9778


Total Biz IDs: 5203


In [3]:
######## Run on all businesses #######

# List of unique business IDs.
#n=1000
all_business_ids = agg_CA['business_id'].unique()
random_n = agg_CA[agg_CA['business_id'].isin(all_business_ids)].copy()
random_n['date_index'] = pd.to_datetime(random_n['date'])
random_n.set_index('date_index', inplace=True)

random_n_vadar=random_n.copy()[['business_id','avg_VADAR','business_name']]
business_ids = random_n_vadar['business_id'].unique()
sample_freq='6M'

# Initialize a variable to keep track of overall accuracy.
overall_accuracy = []

test_predictions_df = pd.DataFrame()
predictions_df=pd.DataFrame()

# Group data by index
for i,business_id in enumerate(business_ids):

    business_data = random_n_vadar[random_n_vadar['business_id'] == business_id]
    
    # Group data by the half_year period and calculate the mean to reduce noise
    HalfYear_grouped = business_data.resample(sample_freq).mean(numeric_only=True)

    #Fill NaN values with the average of the previous and following values if there are still NaN
    HalfYear_grouped_filled = HalfYear_grouped.fillna(method='ffill').fillna(method='bfill')
    
    # Skip the business_id if there are NaNs or too few records to run ARIMA
    if HalfYear_grouped_filled['avg_VADAR'].isna().all() or len(HalfYear_grouped_filled)<5:
        continue

    # Calculate the index for the 80% train-test split.
    split_index = int(0.8 * len(HalfYear_grouped_filled ))
    
    # Split the data into training and testing sets.
    train_data = HalfYear_grouped_filled.iloc[:split_index]
    test_data = HalfYear_grouped_filled.iloc[split_index:]

    # Use auto_arima to optimize both trend and order
    trend_values = ['n', 'c', 't', 'ct']
    best_mse=float('inf')
    for trend_value in trend_values:
        best_model = auto_arima(
            train_data,
            seasonal=True,
            stepwise=True,
            suppress_warnings=True,
            trend=trend_value,
            error_action='ignore')

        # Make predictions on the test data using the best model
        test_predictions = best_model.predict(n_periods=len(test_data))

        # Calculate the MSE for hyperparameters selection
        mse_current = mean_squared_error(test_data['avg_VADAR'], test_predictions)

        if mse_current< best_mse:
            best_mse = mse_current
            best_trend = trend_value
            best_order=best_model.get_params()['order']

    print('best trend',best_trend)
    print('best order',best_order)
    print('-------------------------')

    test_predictions_data = pd.DataFrame({
        'Date': test_data.index,
        'Business_ID': business_id,
        'Actual_VADAR': test_data['avg_VADAR'],
        'Predicted_VADAR': test_predictions
    })
    test_predictions_df = pd.concat([test_predictions_df, test_predictions_data], ignore_index=True)

    # Append the accuracy to the overall_accuracy list.
    overall_accuracy.append(best_mse)

    # Make prediction based on all data
    best_model = auto_arima(
            HalfYear_grouped_filled,
            seasonal=True,
            stepwise=True,
            suppress_warnings=True,
            trend=best_trend,
            error_action='ignore')
    predictions = best_model.predict(n_periods=1)
    
    predictions_data = pd.DataFrame({
        'Business_ID': business_id,
        'Date': HalfYear_grouped_filled.index[-1]+ pd.DateOffset(months=6),
        'Predicted_VADAR': predictions
    })
    predictions_df = pd.concat([predictions_df, predictions_data], ignore_index=True)

# Calculate the average accuracy across all businesses.
average_accuracy = np.mean(overall_accuracy)
print(f"Average Mean Squared Error (MSE) for random {n} businesses: {average_accuracy}")

display(test_predictions_df.head(5))
display(predictions_df.head(5))


best trend c
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 1, 2)
-------------------------
best trend n
best order (0, 1, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend c
best order (1, 1, 0)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (1, 1, 0)
-------------------------
best trend ct
best order (1, 1, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 2)
-------------------------
best trend c
best order (0, 0, 0)

C:\Users\Julie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
C:\Users\Julie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
C:\Users\Julie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
C:\Users\Julie\AppData\Local\Packages\PythonSoftwareFoundation.Pyth

best trend ct
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 3)
-------------------------
best trend n
best order (2, 0, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (2, 0, 0)
-------------------------
best trend t
best order (1, 0, 1)
-------------------------
best trend c
best order (2, 0, 2)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 1, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend c
best order (0, 0, 2)
--

,Date,Business_ID,Actual_VADAR,Predicted_VADAR
0,2019-11-30,-3AooxIkg38UyUdlz5oXdw,0.516794,0.649382
1,2020-05-31,-3AooxIkg38UyUdlz5oXdw,0.679253,0.652768
2,2020-11-30,-3AooxIkg38UyUdlz5oXdw,0.430407,0.656155
3,2021-05-31,-3AooxIkg38UyUdlz5oXdw,0.081177,0.659541
4,2021-11-30,-3AooxIkg38UyUdlz5oXdw,0.012669,0.662928


,Business_ID,Date,Predicted_VADAR
0,-3AooxIkg38UyUdlz5oXdw,2022-11-30,0.563446
1,-6L_z3ftD1iepJb0FfJghw,2017-05-30,0.953165
2,-FM4CxOg4XXmX_Ebky_SiQ,2022-10-30,0.634165
3,-FRHYI6doGCjIsXiKHkUqw,2022-08-28,0.118461
4,-Fka99c-tJ-epWYNIobqyQ,2019-09-30,0.507562


In [4]:
print('Total Num of Test Predictions:',len(test_predictions_df))
print('Test Prediction, Num of Vadar >1:',len(test_predictions_df[(test_predictions_df['Predicted_VADAR']<-1)]))
print('Test Prediction, Num of Vadar <-1:',len(test_predictions_df[(test_predictions_df['Predicted_VADAR']>1)]))
print('--------------------')
print(' ')

print('Num of biz IDs used in prediction: ',len(predictions_df['Business_ID'].unique()))
print('Num of skipped biz IDs due to insufficient info: ',n-len(predictions_df['Business_ID'].unique()))
print('Overall Prediction, Num of Vadar >1:',len(predictions_df[(predictions_df['Predicted_VADAR']<-1)]))
print('Overall Prediction, Num of Vadar <-1:',len(predictions_df[(predictions_df['Predicted_VADAR']>1)]))


Total Num of Test Predictions: 3062
Test Prediction, Num of Vadar >1: 11
Test Prediction, Num of Vadar <-1: 143
--------------------
 
Num of biz IDs used in prediction:  657
Num of skipped biz IDs due to insufficient info:  343
Overall Prediction, Num of Vadar >1: 1
Overall Prediction, Num of Vadar <-1: 26


In [ ]:
test_predictions_df.to_csv('../Data/test_predictions_CA.csv')
predictions_df.to_csv('../Data/predictions_CA.csv')